<div align = "center">

# ***Water Consumption by Governorate Data Cleaning***

</div>

## ***Importing Libraries***

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

## ***Data Loading & Overview:***

### 1. **Loading the Dataset**

In [31]:
def load_data(file_path):
    """ load the dataset from csv file and return dataframe """
    try:
        data = pd.read_csv(file_path)
    except Exception as e:
        print(f"file not found error: {e}")
        return None
    else:
        print("data loaded successfully")
        return data
    finally:
        print("Execution completed.")

file_path = "../Data/Raw Data/Water Consumption by GOV.csv"
df = load_data(file_path)
df.head()

data loaded successfully
Execution completed.


,Gov,Water_Source,Season,Year,Water_Consumption
0,Luxor,Surface_Water,Winter,2011,70703.0
1,Fayoum,Surface_Water,Winter,2011,784952.0
2,Kafr El Sheikh,Surface_Water,Winter,2011,899221.0
3,Bani Suef,Surface_Water,Winter,2011,497020.0
4,Port Said,Surface_Water,Winter,2011,NaN


### 2. ***Data Overview***

In [32]:
# data shape
print(f"Data shape: {df.shape[0]} rows and {df.shape[1]} columns")

#data info & data types
df.info()

Data shape: 1404 rows and 5 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1404 entries, 0 to 1403
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gov                1404 non-null   object 
 1   Water_Source       1404 non-null   object 
 2   Season             1404 non-null   object 
 3   Year               1404 non-null   int64  
 4   Water_Consumption  1359 non-null   float64
dtypes: float64(1), int64(1), object(3)
memory usage: 55.0+ KB


In [33]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Year,1404.0,2017.000000,3.742991,2011.0,2014.0,2017.0,2020.0,2023.0
Water_Consumption,1359.0,376099.209051,581939.006084,0.0,17468.0,111527.0,508808.0,3495271.0


In [34]:
df.describe(include='object').T

,count,unique,top,freq
Gov,1404,26,Giza,104
Water_Source,1404,2,Surface_Water,1144
Season,1404,4,Winter,351


## ***Data Quality Check & Cleaning***

### 1. ***Data Quality Check***

#### **Missing Values**

In [35]:
print("\n--- Missing Values (NaN) ---")
print(df.isna().sum())

print("\n--- Zero Values Count ---")
print((df['Water_Consumption'] == 0).sum())


--- Missing Values (NaN) ---
Gov                   0
Water_Source          0
Season                0
Year                  0
Water_Consumption    45
dtype: int64

--- Zero Values Count ---
41


In [36]:
print("Governorates with missing values (NaN):")
print(df[df['Water_Consumption'].isna()]['Gov'].value_counts())

zero_count = (df['Water_Consumption'] == 0).sum()
print(f"\nNumber of Zero Consumption Rows: {zero_count}")

print("\nZero values distribution across Governorates:")
print(df[df['Water_Consumption'] == 0]['Gov'].value_counts())

Governorates with missing values (NaN):
Gov
Port Said         23
Marsa Matrouh     12
Suez               2
Kafr El Sheikh     2
Luxor              1
Sohag              1
New Vally          1
North Sinai        1
South Sinai        1
Giza               1
Name: count, dtype: int64

Number of Zero Consumption Rows: 41

Zero values distribution across Governorates:
Gov
South Sinai       13
Port Said          5
Sohag              4
Damietta           4
Cairo              4
Kafr El Sheikh     3
Aswan              3
Ismailia           2
Suez               2
Alexandria         1
Name: count, dtype: int64


In [37]:
missing_data = df[df['Water_Consumption'].isnull()]

missing_data.sort_values(
    by=['Gov', 'Season', 'Year'])

,Gov,Water_Source,Season,Year,Water_Consumption
1160,Giza,Groundwater,Perennial,2014,NaN
812,Kafr El Sheikh,Surface_Water,Nile season,2015,NaN
920,Kafr El Sheikh,Surface_Water,Nile season,2019,NaN
729,Luxor,Surface_Water,Nile season,2012,NaN
725,Marsa Matrouh,Groundwater,Nile season,2011,NaN
752,Marsa Matrouh,Groundwater,Nile season,2012,NaN
779,Marsa Matrouh,Groundwater,Nile season,2013,NaN
806,Marsa Matrouh,Groundwater,Nile season,2014,NaN
833,Marsa Matrouh,Groundwater,Nile season,2015,NaN
860,Marsa Matrouh,Groundwater,Nile season,2016,NaN


In [38]:
print("\nNegative Values:")
print((df['Water_Consumption'] < 0).sum())


Negative Values:
0


#### **Duplicated Rows**

In [39]:
print("Full duplicates:", df.duplicated().sum())
print("Duplicate business keys:", 
    df.duplicated(
        subset=['Gov', 'Year', 'Season', "Water_Source"]).sum())

Full duplicates: 0
Duplicate business keys: 0


In [40]:
print("Duplicate Rows Count:", df.duplicated().sum())

print("\nUnique Governorates Count:", df['Gov'].nunique())
print("Governors List:", df['Gov'].unique())

print("\nWater Sources:", df['Water_Source'].unique())
print("Seasons:", df['Season'].unique())
print("Years Range:", df['Year'].min(), "to", df['Year'].max())

Duplicate Rows Count: 0

Unique Governorates Count: 26
Governors List: ['Luxor' 'Fayoum' 'Kafr El Sheikh' 'Bani Suef' 'Port Said' 'Damietta'
 'Ismailia' 'Monufia' 'Sohag' 'Minya' 'Suez' 'Qena' 'Sharqiya' 'Asyut'
 'Aswan' 'Beheira' 'Dakahlia' 'Qalyoubia' 'Giza' 'Gharbiya' 'Cairo'
 'Alexandria' 'New Vally' 'Marsa Matrouh' 'North Sinai' 'South Sinai']

Water Sources: ['Surface_Water' 'Groundwater']
Seasons: ['Winter' 'Summer' 'Nile season' 'Perennial']
Years Range: 2011 to 2023


### 2. ***Data Cleaning***

In [41]:
df_cleaned = df.copy()

In [42]:
df_cleaned.columns = (
    df_cleaned.columns.str.strip()
    .str.replace(" ","_", regex=False)
    .str.replace("-","_", regex=False)
    .str.replace("/","_", regex=False)
    .str.lower())

print(df_cleaned.columns)

Index(['gov', 'water_source', 'season', 'year', 'water_consumption'], dtype='object')


In [43]:
df_cleaned.to_csv("../Data/Clean Data/Water_Consumption_GOV_clean.csv", index=False)